<a href="https://colab.research.google.com/github/amol004/Flask-Demo/blob/main/Flask_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
from datetime import datetime
import datetime as dt
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import cross_validate
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.model_selection import RandomizedSearchCV , cross_val_score, KFold
from sklearn import metrics
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error
from sklearn.metrics import accuracy_score
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import log_loss
import warnings
warnings.filterwarnings('ignore')

In [3]:
# Load Dataset
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
# importing user dataset
data = pd.read_csv('/content/drive/MyDrive/Crop_recommendation.csv', encoding='latin1')

In [8]:
data

,N,P,K,temperature,humidity,ph,rainfall,label
0,90,42,43,20.879744,82.002744,6.502985,202.935536,rice
1,85,58,41,21.770462,80.319644,7.038096,226.655537,rice
2,60,55,44,23.004459,82.320763,7.840207,263.964248,rice
3,74,35,40,26.491096,80.158363,6.980401,242.864034,rice
4,78,42,42,20.130175,81.604873,7.628473,262.717340,rice
...,...,...,...,...,...,...,...,...
2195,107,34,32,26.774637,66.413269,6.780064,177.774507,coffee
2196,99,15,27,27.417112,56.636362,6.086922,127.924610,coffee
2197,118,33,30,24.131797,67.225123,6.362608,173.322839,coffee
2198,117,32,34,26.272418,52.127394,6.758793,127.175293,coffee


In [12]:
# import pandas as pd
# from sklearn.preprocessing import StandardScaler
# from sklearn.ensemble import RandomForestClassifier
# from sklearn.model_selection import train_test_split

# #
# # Load the csv file
# df = pd.read_csv("Crop_recommendation.csv")
# #
# print(df.head())
#
# X = df[["N", "P", "K", "temperature","humidity","ph","rainfall"]]
# y = df["label"]
#
# # Split the dataset into train and test
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=50)
#
# # Feature scaling
# sc = StandardScaler()
# X_train = sc.fit_transform(X_train)
# X_test= sc.transform(X_test)
#
# # Instantiate the model
# classifier = RandomForestClassifier()
#
# # Fit the model
# classifier.fit(X_train, y_train)
#
# # Make pickle file of our model
# pickle.dump(classifier, open("model.pkl", "wb"))

# Import necessary libraries
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
import pickle
# Load the dataset


# Split the data into features and labels
X = data.iloc[:, :-1]  # Features
y = data.iloc[:, -1]   # Labels

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Create the model
model = RandomForestClassifier()

# Train the model
model.fit(X_train, y_train)

# Make predictions on test data
# predictions = model.predict(X_test)

#pickle.dump(model, open("model.pkl", "wb"))
# Evaluate the model
accuracy = model.score(X_test, y_test)
print("Accuracy:", accuracy)

# Example usage: Predict crop for a new set of features
new_features = [[117 ,32,34,26.2724184,52.12739421,6.758792552,127.1752928]]  # Replace with your own set of features
predicted_crop = model.predict(new_features)
print("Predicted crop:", predicted_crop)


Accuracy: 0.9931818181818182
Predicted crop: ['coffee']


In [13]:
pickle.dump(model, open("model.pkl", "wb"))
print("Model saved as model.pkl")

Model saved as model.pkl


In [49]:
%%writefile app.py
from flask import Flask, request, jsonify, render_template
import pickle
import numpy as np
import os

app1 = Flask(__name__)

# Load the trained model
model = pickle.load(open('model.pkl', 'rb'))

@app1.route('/')
def home():
    return render_template('index.html')

@app1.route('/predict', methods=['POST'])
def predict():
    if request.is_json:
        # Handle JSON input for API calls
        data = request.get_json(force=True)
        new_features = np.array(data['features']).reshape(1, -1)
        prediction = model.predict(new_features)
        return jsonify(prediction[0])
    else:
        # Handle form input for web predictions
        try:
            N = float(request.form['N'])
            P = float(request.form['P'])
            K = float(request.form['K'])
            temperature = float(request.form['temperature'])
            humidity = float(request.form['humidity'])
            ph = float(request.form['ph'])
            rainfall = float(request.form['rainfall'])

            new_features = np.array([[N, P, K, temperature, humidity, ph, rainfall]])
            prediction = model.predict(new_features)
            return render_template('index.html', prediction_text=f'The recommended crop is: {prediction[0]}')
        except Exception as e:
            return render_template('index.html', prediction_text=f'Error processing input: {e}')

if __name__ == '__main__':
    # Ensure the templates directory exists
    if not os.path.exists('templates'):
        os.makedirs('templates')
    app1.run(debug=True, host='0.0.0.0', port=5008)

Overwriting app.py


In [23]:
!pip install pyngrok

# Task
Deploy the `app.py` Flask application, which uses the `model.pkl` file, by terminating any existing Flask processes on port 5008, running the application in the background, and exposing it via an ngrok tunnel. Then, provide instructions to test the deployed application through its web form and the `/predict` JSON API endpoint using the generated ngrok public URL.

## Terminate Flask Process

### Subtask:
Terminate any existing Python processes that are running on port 5008 to ensure a clean restart of the Flask application.


**Reasoning**:
To ensure a clean restart of the Flask application, I will terminate any existing processes running on port 5008. This involves finding the process IDs (PIDs) associated with that port and then killing them if any are found.



In [51]:
import subprocess

# Find PIDs of processes using port 5008
find_pids_command = 'lsof -t -i:5008'
try:
    pids_output = subprocess.check_output(find_pids_command, shell=True, text=True)
    pids = pids_output.strip().split('\n') if pids_output.strip() else []

    if pids:
        print(f"Found processes on port 5008: {pids}")
        for pid in pids:
            command = f'kill -9 {pid}'
            try:
                subprocess.run(command, shell=True, check=True)
                print(f"Terminated process {pid}")
            except subprocess.CalledProcessError as e:
                print(f"Error terminating process {pid}: {e}")
    else:
        print("No processes found on port 5008.")
except subprocess.CalledProcessError as e:
    print(f"Error finding processes on port 5008: {e}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")


Error finding processes on port 5008: Command 'lsof -t -i:5008' returned non-zero exit status 1.


In [52]:
import subprocess

# Run the Flask app in the background, redirecting output to app.log
flask_command = "nohup python app.py > app.log 2>&1 &"
subprocess.Popen(flask_command, shell=True)

print("Flask application is starting in the background. Check app.log for details.")

Flask application is starting in the background. Check app.log for details.


In [53]:
import time

# Give the Flask app a moment to start and write to the log
time.sleep(5)

# Display the content of app.log to verify startup
with open('app.log', 'r') as f:
    log_content = f.read()
    print(log_content)

# Optional: Check for specific success messages or errors in the log_content

 * Serving Flask app 'app'
 * Debug mode: on
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5008
 * Running on http://172.28.0.12:5008
Press CTRL+C to quit
 * Restarting with watchdog (inotify)
 * Debugger is active!
 * Debugger PIN: 346-595-961



In [55]:
from pyngrok import ngrok
import os

# Replace 'YOUR_AUTHTOKEN' with your actual ngrok authtoken
# You can get one from https://dashboard.ngrok.com/auth
NGROK_AUTH_TOKEN = "2hd4M75y93zZp4H74YxO1nFv94R_4iV2M5c165q3M5s35f6Y0" # Directly assign the authtoken

if not NGROK_AUTH_TOKEN or NGROK_AUTH_TOKEN == "YOUR_AUTHTOKEN":
    print("Please set your NGROK_AUTH_TOKEN environment variable or replace 'YOUR_AUTHTOKEN' in the code.")
else:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)

    # Start a ngrok tunnel for the Flask app on port 5008
    # The 'app' should be running on this port
    public_url = ngrok.connect(5008)
    print(f"Ngrok tunnel for Flask app available at: {public_url}")
    print("You can now access your Flask application through this public URL.")
    print("To test the /predict endpoint, send a POST request to {public_url}/predict with JSON data.")
    print("For example, using curl:")
    print(f"curl -X POST -H \"Content-Type: application/json\" -d \'{{\"features\": [117, 32, 34, 26.2724184, 52.12739421, 6.758792552, 127.1752928]}}\' {public_url}/predict")


Ngrok tunnel for Flask app available at: NgrokTunnel: "https://ciera-unpeppery-brendan.ngrok-free.dev" -> "http://localhost:5008"
You can now access your Flask application through this public URL.
To test the /predict endpoint, send a POST request to {public_url}/predict with JSON data.
For example, using curl:
curl -X POST -H "Content-Type: application/json" -d '{"features": [117, 32, 34, 26.2724184, 52.12739421, 6.758792552, 127.1752928]}' NgrokTunnel: "https://ciera-unpeppery-brendan.ngrok-free.dev" -> "http://localhost:5008"/predict
